### RAG Pipilines - Data Ingestion to VecotrDB pipline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\shahk\Documents\code\RAG-Langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Google Agentic AI Hack.pdf
  ✓ Loaded 18 pages

Processing: Paper14871.pdf
  ✓ Loaded 4 pages

Processing: resume.pdf
  ✓ Loaded 1 pages

Total documents loaded: 23


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-07-27T05:07:58+00:00', 'title': 'Google Agentic AI Hack', 'moddate': '2025-07-27T05:07:56+00:00', 'keywords': 'DAGsaf3fInM,BAD9z7kn8Tg,0', 'author': 'Kushal Shah', 'source': '..\\data\\pdf\\Google Agentic AI Hack.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Google Agentic AI Hack.pdf', 'file_type': 'pdf'}, page_content='TEAM DETAILS\nTeam Name - Majdoor AI\nTeam Leader Name - Kushal Shah \nProblem Statement - Receipt Management for Google Wallet'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-07-27T05:07:58+00:00', 'title': 'Google Agentic AI Hack', 'moddate': '2025-07-27T05:07:56+00:00', 'keywords': 'DAGsaf3fInM,BAD9z7kn8Tg,0', 'author': 'Kushal Shah', 'source': '..\\data\\pdf\\Google Agentic AI Hack.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2', 'source_file': 'Google Agentic AI Hack.pdf', 'file_type': 'pdf'}, page_content='I D E A

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    


    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 23 into 45 chunks

Example chunk:
Content: TEAM DETAILS
Team Name - Majdoor AI
Team Leader Name - Kushal Shah 
Problem Statement - Receipt Management for Google Wallet...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-07-27T05:07:58+00:00', 'title': 'Google Agentic AI Hack', 'moddate': '2025-07-27T05:07:56+00:00', 'keywords': 'DAGsaf3fInM,BAD9z7kn8Tg,0', 'author': 'Kushal Shah', 'source': '..\\data\\pdf\\Google Agentic AI Hack.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Google Agentic AI Hack.pdf', 'file_type': 'pdf'}


In [6]:
print(f"\nTotal chunks created: {len(chunks)}")


Total chunks created: 45


### Embedding and vecotrStoreDB

In [1]:
import numpy as np
# from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'chromadb'

In [2]:
class EmbeddingManager:
    def  __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager with a specified model.

        Args:
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

        